# 📊 Volume & Trend Analysis — Improved Version v2

**การปรับปรุงจาก v1:**
- ✅ Cell 1: RVOL ครอบคลุมทั้งวัน + กรอง outlier (Earnings days)
- ✅ Cell 2: Fibonacci ใช้ window เดียวกัน + Weighted Score
- ✅ Cell 3 (ใหม่): RSI Divergence + OBV Trend + ATR-based Stop Loss
- ✅ Cell 4 (ใหม่): สรุปภาพรวมทุก indicator รวมเป็น Final Signal เดียว

In [ ]:
# ============================================================
#  SHARED CONFIG — แก้ตรงนี้ที่เดียวพอ
# ============================================================
STOCK_LIST = ['SNDK', 'COHR', 'LITE', 'OXY', 'COP']

import yfinance as yf
import pandas as pd
import numpy as np
import datetime
import warnings
warnings.filterwarnings('ignore')

# ── Helper: ดึง Series จาก yfinance ไม่ว่าจะเป็น MultiIndex หรือไม่ ──
def _get_series(df: pd.DataFrame, col: str, ticker: str) -> pd.Series:
    """ดึง Series จาก DataFrame ที่อาจเป็น MultiIndex columns"""
    if isinstance(df.columns, pd.MultiIndex):
        data = df[col]
        if isinstance(data, pd.DataFrame):
            return data[ticker].dropna() if ticker in data.columns else data.iloc[:, 0].dropna()
        return data.dropna()
    return df[col].dropna()

print("✅ Config โหลดเรียบร้อย — หุ้นที่จะวิเคราะห์:", STOCK_LIST)

---
## 📦 Cell 1 — RVOL Analysis (Full Day + Outlier Filter)

**สิ่งที่แก้ไขจาก v1:**
1. วิเคราะห์ volume ทั้งวัน (09:30–16:00) ไม่ใช่แค่ 90 นาทีแรก
2. กรองวัน outlier (volume สูงกว่าค่าเฉลี่ย 3x) ออกก่อนคำนวณ average — ป้องกัน earnings/split เบี้ยวค่า
3. เพิ่ม Intraday Volume Profile: แสดงว่า volume วันนี้กระจุกช่วงไหน

In [ ]:
def calculate_rvol_fullday(tickers, period='15d'):
    results = []

    for ticker in tickers:
        print(f"กำลังดึงข้อมูล: {ticker}...")
        try:
            df = yf.download(ticker, period=period, interval='15m', progress=False)
            if df.empty:
                print(f"  ⚠️  ไม่พบข้อมูลของ {ticker}")
                continue

            df = df.reset_index()
            time_col = df.columns[0]

            # Timezone → US/Eastern
            if df[time_col].dt.tz is None:
                df[time_col] = df[time_col].dt.tz_localize('UTC').dt.tz_convert('US/Eastern')
            else:
                df[time_col] = df[time_col].dt.tz_convert('US/Eastern')

            # ✅ FIX: ใช้ทั้งวัน 09:30–16:00 (ไม่ใช่แค่ 90 นาที)
            df['Time'] = df[time_col].dt.time
            df_day = df[
                (df['Time'] >= datetime.time(9, 30)) &
                (df['Time'] < datetime.time(16, 0))
            ].copy()
            df_day['Date'] = df_day[time_col].dt.date

            # ดึง Volume Series
            vol_col_data = df_day['Volume'] if 'Volume' in df_day.columns else None
            if vol_col_data is None:
                continue
            if isinstance(vol_col_data, pd.DataFrame):
                df_day['_vol'] = vol_col_data[ticker] if ticker in vol_col_data.columns else vol_col_data.iloc[:, 0]
            else:
                df_day['_vol'] = vol_col_data

            vol_series = df_day.groupby('Date')['_vol'].sum()

            if len(vol_series) < 3:
                print(f"  ⚠️  ข้อมูลของ {ticker} มีน้อยเกินไป")
                continue

            dates = sorted(vol_series.index)
            today = dates[-1]
            past_vols = vol_series[dates[:-1]]

            # ✅ FIX: กรอง outlier ออก (วัน volume > 3x median = earnings/split)
            median_vol = past_vols.median()
            clean_past_vols = past_vols[past_vols <= 3 * median_vol]
            outlier_days = len(past_vols) - len(clean_past_vols)

            today_vol = float(vol_series[today])
            avg_past_vol = float(clean_past_vols.mean())
            rvol = today_vol / avg_past_vol if avg_past_vol > 0 else 0

            # ✅ ใหม่: Intraday Volume Profile (ช่วงไหน volume หนาแน่นที่สุดวันนี้)
            today_intraday = df_day[df_day['Date'] == today].copy()
            today_intraday['Hour'] = today_intraday[time_col].dt.hour
            vol_by_hour = today_intraday.groupby('Hour')['_vol'].sum()
            peak_hour = int(vol_by_hour.idxmax()) if not vol_by_hour.empty else 9
            peak_label = f"{peak_hour}:00–{peak_hour+1}:00"

            # Signal
            if rvol >= 2.0:
                signal = '🔥 Extreme Volume (>2x avg)'
            elif rvol >= 1.5:
                signal = '✅ Strong Reversal'
            elif rvol >= 0.8:
                signal = '⬜ Normal / Sideways'
            else:
                signal = '🔴 Bearish Continuation (Volume แห้ง)'

            results.append({
                'Ticker': ticker,
                'Date': today.strftime('%Y-%m-%d'),
                'Volume_Today': f"{int(today_vol):,}",
                'Avg_Volume (clean)': f"{int(avg_past_vol):,}",
                'Outlier_Days_Removed': outlier_days,
                'RVOL': round(rvol, 2),
                'Peak_Hour': peak_label,
                'Signal': signal
            })

        except Exception as e:
            print(f"  ❌ Error {ticker}: {e}")

    return pd.DataFrame(results)


# ── RUN ──
df_rvol = calculate_rvol_fullday(STOCK_LIST)
print("\n" + "="*65)
print("  RVOL Analysis — Full Day (with Outlier Filter)")
print("="*65)
print(df_rvol.to_string(index=False))

---
## 📐 Cell 2 — Pullback vs Bearish Reversal (Weighted Score)

**สิ่งที่แก้ไขจาก v1:**
1. `recent_high` และ `recent_low` ใช้ window เดียวกัน (60 วัน) → Fibonacci แม่นขึ้น
2. Volume analysis ดู 10 วัน (จาก 5) และเปรียบเทียบกับ median แทน mean → robust กว่า
3. Weighted Score: EMA50 = 3 คะแนน, Fib = 2 คะแนน, Volume = 1 คะแนน (สะท้อนความสำคัญจริง)

In [ ]:
def check_pullback_or_bearish_v2(tickers):
    results = []

    for ticker in tickers:
        try:
            df = yf.download(ticker, period='6mo', interval='1d', progress=False)
            if df.empty:
                continue

            close = _get_series(df, 'Close', ticker)
            high  = _get_series(df, 'High',  ticker)
            low   = _get_series(df, 'Low',   ticker)
            vol   = _get_series(df, 'Volume',ticker)

            current_price = float(close.iloc[-1])

            # ── Indicator 1: EMA 50 (น้ำหนัก 3) ──
            ema_50 = float(close.ewm(span=50, adjust=False).mean().iloc[-1])
            above_ema50 = current_price > ema_50

            # ── Indicator 2: Fibonacci 61.8% (น้ำหนัก 2) ──
            # ✅ FIX: ใช้ window เดียวกัน 60 วัน ทั้ง high และ low
            window = 60
            recent_high = float(high.tail(window).max())
            recent_low  = float(low.tail(window).min())
            fib_618 = recent_high - (0.618 * (recent_high - recent_low))
            above_fib618 = current_price > fib_618

            # ── Indicator 3: Volume Quality (น้ำหนัก 1) ──
            # ✅ FIX: ดู 10 วัน และใช้ median เป็น baseline (robust ต่อ outlier)
            lookback = 10
            median_vol_20 = float(vol.tail(20).median())
            ret_10d = close.pct_change().tail(lookback)
            down_vol = vol.tail(lookback)[ret_10d < 0]
            high_sell_vol = bool(down_vol.mean() > median_vol_20) if len(down_vol) > 0 else False

            # ── Weighted Score ──
            score  = (3 if above_ema50 else 0)     # EMA50 สำคัญที่สุด
            score += (2 if above_fib618 else 0)    # Fib สำคัญรองลงมา
            score += (1 if not high_sell_vol else 0)  # Volume ช่วยยืนยัน
            max_score = 6

            if score >= 5:
                status = "🟢 Pullback (โครงสร้างแข็งแกร่ง)"
            elif score >= 3:
                status = "🟡 Warning (เริ่มเสี่ยง ต้องระวัง)"
            elif score >= 1:
                status = "🟠 High Risk (โครงสร้างพัง)"
            else:
                status = "🔴 Bearish Reversal (เทรนเปลี่ยนชัดเจน)"

            results.append({
                'Ticker':         ticker,
                'Price':          f"{current_price:.2f}",
                'EMA_50':         f"{ema_50:.2f}",
                '>EMA50 (x3)':    '✅' if above_ema50 else '❌',
                'Fib_61.8%':      f"{fib_618:.2f}",
                '>Fib (x2)':      '✅' if above_fib618 else '❌',
                'High_SellVol':   '⚠️ Yes' if high_sell_vol else 'No',
                'Score':          f"{score}/{max_score}",
                'Status':         status
            })

        except Exception as e:
            print(f"❌ Error {ticker}: {e}")

    return pd.DataFrame(results)


# ── RUN ──
df_pullback = check_pullback_or_bearish_v2(STOCK_LIST)
print("\n" + "="*65)
print("  Pullback vs Bearish Reversal — Weighted Score")
print("="*65)
print(df_pullback.to_string(index=False))

---
## 📈 Cell 3 — Momentum Confirmation (RSI + OBV + ATR)

**Cell ใหม่ทั้งหมด — 3 Indicators:**
- **RSI (14)**: ดู oversold (< 35) และ RSI Divergence (ราคาลงแต่ RSI ไม่ลง = ขาขึ้นซ่อนอยู่)
- **OBV Trend**: On-Balance Volume — ถ้า OBV ขึ้นในขณะที่ราคาย่อ = สัญญาณสะสม (accumulation)
- **ATR Stop**: คำนวณ stop-loss ที่เหมาะสมตาม volatility จริง (ไม่ใช่ fixed %)

In [ ]:
def calculate_momentum_indicators(tickers):
    results = []

    for ticker in tickers:
        try:
            df = yf.download(ticker, period='6mo', interval='1d', progress=False)
            if df.empty:
                continue

            close = _get_series(df, 'Close', ticker)
            high  = _get_series(df, 'High',  ticker)
            low   = _get_series(df, 'Low',   ticker)
            vol   = _get_series(df, 'Volume',ticker)

            current_price = float(close.iloc[-1])

            # ── RSI (14) ──
            delta = close.diff()
            gain  = delta.clip(lower=0).rolling(14).mean()
            loss  = (-delta.clip(upper=0)).rolling(14).mean()
            rs    = gain / loss.replace(0, np.nan)
            rsi   = 100 - (100 / (1 + rs))
            current_rsi = float(rsi.iloc[-1])

            # RSI Divergence: ราคา Low ใหม่ แต่ RSI สูงกว่า Low ก่อนหน้า = Bullish Divergence
            price_lower = float(close.iloc[-1]) < float(close.iloc[-6:-1].min())
            rsi_higher  = float(rsi.iloc[-1])  > float(rsi.iloc[-6:-1].min())
            bullish_divergence = price_lower and rsi_higher

            if current_rsi < 30:
                rsi_signal = '🔥 Oversold (<30)'
            elif current_rsi < 40:
                rsi_signal = '⚠️ Near Oversold'
            elif current_rsi > 70:
                rsi_signal = '⛔ Overbought'
            else:
                rsi_signal = '⬜ Neutral'

            # ── OBV Trend ──
            obv = (np.sign(close.diff()) * vol).fillna(0).cumsum()
            # OBV EMA 10 เพื่อดู trend
            obv_ema = obv.ewm(span=10, adjust=False).mean()
            obv_rising = float(obv_ema.iloc[-1]) > float(obv_ema.iloc[-5])

            # ราคาย่อ แต่ OBV ยังขึ้น = accumulation signal
            price_dropped_5d = float(close.iloc[-1]) < float(close.iloc[-5])
            accumulation = price_dropped_5d and obv_rising

            obv_signal = '🟢 Accumulation (สะสม)' if accumulation else (
                         '🔵 OBV Rising' if obv_rising else '🔴 Distribution (ขายออก)')

            # ── ATR-based Stop Loss (14) ──
            prev_close = close.shift(1)
            tr = pd.concat([
                high - low,
                (high - prev_close).abs(),
                (low  - prev_close).abs()
            ], axis=1).max(axis=1)
            atr = tr.rolling(14).mean()
            current_atr = float(atr.iloc[-1])

            # Stop = ราคาปัจจุบัน - 2x ATR (conservative trailing stop)
            stop_price = current_price - (2 * current_atr)
            stop_pct   = (current_price - stop_price) / current_price * 100

            results.append({
                'Ticker':          ticker,
                'Price':           f"{current_price:.2f}",
                'RSI_14':          f"{current_rsi:.1f}",
                'RSI_Signal':      rsi_signal,
                'Bullish_Div':     '✅ Yes' if bullish_divergence else 'No',
                'OBV_Signal':      obv_signal,
                'ATR_14':          f"{current_atr:.2f}",
                'ATR_Stop':        f"{stop_price:.2f} (-{stop_pct:.1f}%)"  
            })

        except Exception as e:
            print(f"❌ Error {ticker}: {e}")

    return pd.DataFrame(results)


# ── RUN ──
df_momentum = calculate_momentum_indicators(STOCK_LIST)
print("\n" + "="*65)
print("  Momentum Confirmation — RSI + OBV + ATR Stop")
print("="*65)
print(df_momentum.to_string(index=False))

---
## 🎯 Cell 4 — Final Composite Signal (รวมทุก Indicator)

**Cell ใหม่: สรุปภาพรวมจาก 3 Cell ข้างต้น → Final Score + Action**

| Signal | Score | ความหมาย |
|--------|-------|----------|
| 🟢 Strong Buy | 8–10 | โอกาสดีมาก ทุก indicator ยืนยัน |
| 🔵 Watch | 5–7  | น่าสนใจ แต่รอยืนยันเพิ่ม |
| 🟡 Neutral | 3–4  | ยังไม่ชัดเจน |
| 🔴 Avoid | 0–2  | Risk สูง ยังไม่ควรเข้า |

In [ ]:
def build_final_signal(df_rvol, df_pullback, df_momentum):
    final_rows = []

    all_tickers = df_rvol['Ticker'].tolist() if not df_rvol.empty else []

    for ticker in all_tickers:
        score = 0
        notes = []

        # ── RVOL Score (max 3 คะแนน) ──
        r = df_rvol[df_rvol['Ticker'] == ticker]
        if not r.empty:
            rvol_val = float(r['RVOL'].values[0])
            if rvol_val >= 2.0:
                score += 3; notes.append(f"RVOL={rvol_val:.2f}🔥")
            elif rvol_val >= 1.5:
                score += 2; notes.append(f"RVOL={rvol_val:.2f}✅")
            elif rvol_val >= 0.8:
                score += 1; notes.append(f"RVOL={rvol_val:.2f}⬜")
            else:
                notes.append(f"RVOL={rvol_val:.2f}🔴")

        # ── Pullback Score (max 3 คะแนน จาก Weighted Score ของ Cell 2) ──
        p = df_pullback[df_pullback['Ticker'] == ticker]
        if not p.empty:
            raw_score_str = p['Score'].values[0]  # เช่น "5/6"
            raw = int(raw_score_str.split('/')[0])
            pullback_pts = round(raw / 6 * 3)     # normalize → 0–3
            score += pullback_pts
            notes.append(f"Struct={raw_score_str}")

        # ── Momentum Score (max 4 คะแนน) ──
        m = df_momentum[df_momentum['Ticker'] == ticker]
        if not m.empty:
            rsi_val = float(m['RSI_14'].values[0])
            div     = m['Bullish_Div'].values[0]
            obv_sig = m['OBV_Signal'].values[0]

            if rsi_val < 35:
                score += 2; notes.append("RSI_Oversold✅")
            elif rsi_val < 45:
                score += 1; notes.append("RSI_Low⚠️")

            if 'Yes' in str(div):
                score += 1; notes.append("BullDiv✅")

            if 'Accumulation' in obv_sig:
                score += 1; notes.append("OBV_Accum✅")
            elif 'Rising' in obv_sig:
                score += 0; notes.append("OBV_Rise⬜")
            else:
                notes.append("OBV_Dist🔴")

        # ── Final Signal ──
        if score >= 8:
            final = "🟢 Strong Buy"
        elif score >= 5:
            final = "🔵 Watch & Wait"
        elif score >= 3:
            final = "🟡 Neutral"
        else:
            final = "🔴 Avoid"

        final_rows.append({
            'Ticker':       ticker,
            'Total_Score':  f"{score}/10",
            'Final_Signal': final,
            'Key_Notes':    ' | '.join(notes)
        })

    return pd.DataFrame(final_rows).sort_values('Total_Score', ascending=False)


# ── RUN ──
df_final = build_final_signal(df_rvol, df_pullback, df_momentum)
print("\n" + "="*75)
print("  🎯 FINAL COMPOSITE SIGNAL")
print("="*75)
print(df_final.to_string(index=False))
print("\n📌 หมายเหตุ: ใช้เป็น input ประกอบการตัดสินใจเท่านั้น ไม่ใช่คำแนะนำการลงทุน")

---
## 💬 Cell 5 — สรุปภาษาคนธรรมดา (Plain Language Summary)

**อ่านง่าย ไม่ต้องมีพื้นฐาน finance** — ตอบ 3 คำถามหลักต่อหุ้น:
1. **ขึ้นของจริง หรือ แค่เด้งก่อนลงต่อ?**
2. **ถ้ายังถืออยู่และ "กำไร" → ควรทำอะไร?**
3. **ถ้ายังถืออยู่และ "ขาดทุน" → ควรทำอะไร?**

In [ ]:
def plain_language_summary(df_final, df_pullback, df_momentum, df_rvol):
    """
    แปลผลลัพธ์จากทุก Cell ให้เป็นภาษาพูดธรรมดา
    ตอบ 3 คำถาม: ขึ้นจริงไหม / ถือกำไรทำไง / ถือขาดทุนทำไง
    """

    SEP = "─" * 60

    print("\n" + "═" * 60)
    print("   💬 สรุปภาษาคนธรรมดา — อ่านแล้วตัดสินใจได้เลย")
    print("═" * 60)

    for _, row in df_final.iterrows():
        ticker     = row['Ticker']
        score_str  = row['Total_Score']          # เช่น "7/10"
        score      = int(score_str.split('/')[0])
        signal     = row['Final_Signal']

        # ดึงข้อมูลเพิ่มเติมจากแต่ละ dataframe
        pb_row  = df_pullback[df_pullback['Ticker'] == ticker]
        mo_row  = df_momentum[df_momentum['Ticker'] == ticker]
        rv_row  = df_rvol[df_rvol['Ticker'] == ticker]

        current_price  = float(mo_row['Price'].values[0])   if not mo_row.empty else 0
        atr_stop_str   = mo_row['ATR_Stop'].values[0]       if not mo_row.empty else ""
        stop_price     = float(atr_stop_str.split(' ')[0])  if atr_stop_str else current_price * 0.95
        rsi_val        = float(mo_row['RSI_14'].values[0])  if not mo_row.empty else 50
        obv_sig        = mo_row['OBV_Signal'].values[0]     if not mo_row.empty else ""
        bull_div       = mo_row['Bullish_Div'].values[0]    if not mo_row.empty else "No"
        struct_score   = pb_row['Score'].values[0]          if not pb_row.empty else "0/6"
        above_ema      = pb_row['>EMA50 (x3)'].values[0]   if not pb_row.empty else "❌"
        above_fib      = pb_row['>Fib (x2)'].values[0]     if not pb_row.empty else "❌"
        rvol_val       = float(rv_row['RVOL'].values[0])    if not rv_row.empty else 0

        # ══════════════════════════════════════════
        # 1. ขึ้นของจริง หรือ แค่เด้งก่อนลงต่อ?
        # ══════════════════════════════════════════
        strong_signals = 0
        weak_signals   = 0

        if above_ema == '✅':   strong_signals += 1
        else:                   weak_signals   += 1
        if above_fib == '✅':   strong_signals += 1
        else:                   weak_signals   += 1
        if rvol_val >= 1.5:     strong_signals += 1
        else:                   weak_signals   += 1
        if 'Accumulation' in obv_sig or 'Rising' in obv_sig:
                                strong_signals += 1
        else:                   weak_signals   += 1
        if 'Yes' in str(bull_div): strong_signals += 1
        if rsi_val < 40:        strong_signals += 1
        else:                   weak_signals   += 1

        if strong_signals >= 5:
            verdict       = "✅ ขึ้นของจริง"
            verdict_detail = ("สัญญาณทุกตัวชี้ไปทิศเดียวกัน — "
                              "มีแรงซื้อจริง โครงสร้างราคายังแข็ง และ volume รองรับ "
                              "โอกาสที่จะขึ้นต่อมีมากกว่าลง")
        elif strong_signals >= 3:
            verdict       = "🟡 กำกวม — น่าจะขึ้น แต่ยังไม่ชัด"
            verdict_detail = ("มีสัญญาณดีบางส่วน แต่ยังมีจุดอ่อนอยู่ด้วย "
                              "อาจเด้งขึ้นระยะสั้นก่อน แล้วค่อยชี้ทิศชัดขึ้น "
                              "ควรรอยืนยันก่อนตัดสินใจหนัก")
        else:
            verdict       = "🔴 เด้งสั้น ก่อนลงต่อ"
            verdict_detail = ("สัญญาณส่วนใหญ่ยังอ่อนแอ — การขึ้นที่เห็นอยู่ "
                              "มักเป็นแค่การเด้งสั้น (dead cat bounce) "
                              "ก่อนที่ราคาจะกลับมาลงต่อ")

        # ══════════════════════════════════════════
        # 2. คำแนะนำถ้าถือหุ้นอยู่แบบ "กำไร"
        # ══════════════════════════════════════════
        if strong_signals >= 5:
            hold_profit = (
                f"📌 ถือต่อได้เลย — แนวโน้มยังดี\n"
                f"   → ลองขยับ Stop Loss ขึ้นมาที่ประมาณ {stop_price:.2f} USD "
                f"(ห่างจากราคาปัจจุบันประมาณ {abs(current_price - stop_price) / current_price * 100:.1f}%)\n"
                f"   → ถ้ากำไรเกิน 20% แล้ว อาจขายออกบางส่วน (เช่น 30%) เพื่อล็อกกำไร"
            )
        elif strong_signals >= 3:
            hold_profit = (
                f"⚠️ ถือต่อได้ แต่ระวังตัว\n"
                f"   → ตั้ง Stop Loss ไว้ที่ {stop_price:.2f} USD — ถ้าหลุดระดับนี้ให้ขายทันที\n"
                f"   → อย่าเพิ่มขนาด position ตอนนี้ รอให้สัญญาณชัดขึ้นก่อน"
            )
        else:
            hold_profit = (
                f"🔔 พิจารณาขายทำกำไรบางส่วนหรือทั้งหมด\n"
                f"   → สัญญาณเริ่มอ่อนแอ — กำไรที่มีอยู่อาจหายได้ถ้ารอต่อ\n"
                f"   → ถ้าไม่อยากขายทั้งหมด ให้ตั้ง Stop ที่ {stop_price:.2f} USD อย่างเคร่งครัด"
            )

        # ══════════════════════════════════════════
        # 3. คำแนะนำถ้าถือหุ้นอยู่แบบ "ขาดทุน"
        # ══════════════════════════════════════════
        if strong_signals >= 5:
            hold_loss = (
                f"💪 สัญญาณดี — ถือต่อได้ถ้ายังทนรับความเสี่ยงได้\n"
                f"   → โอกาสราคาฟื้นตัวกลับมามีอยู่จริง\n"
                f"   → แต่ถ้าราคาหลุดต่ำกว่า {stop_price:.2f} USD ให้หยุดขาดทุน (cut loss) ทันที\n"
                f"   → ห้ามถัวลงถ้ายังไม่มั่นใจ 100%"
            )
        elif strong_signals >= 3:
            hold_loss = (
                f"⚠️ สถานการณ์กำกวม — ควรตัดสินใจตามต้นทุนของตัวเอง\n"
                f"   → ถ้าขาดทุนไม่เกิน 5–8% และยังทนได้ → ถือรอสัญญาณชัดขึ้น\n"
                f"   → ถ้าขาดทุนเกิน 10% ขึ้นไป → ควร cut loss แล้วรอดูใหม่\n"
                f"   → ตั้ง Stop Loss ไว้ที่ {stop_price:.2f} USD อย่างเคร่งครัด"
            )
        else:
            hold_loss = (
                f"🚨 แนะนำ Cut Loss — สัญญาณยังอ่อนแอ\n"
                f"   → การถือต่อมีความเสี่ยงสูงที่จะขาดทุนเพิ่ม\n"
                f"   → ขายออกแล้วรอดูสัญญาณฟื้นตัวที่ชัดขึ้น จะปลอดภัยกว่า\n"
                f"   → ราคาที่ควร cut คือต่ำกว่า {stop_price:.2f} USD (หรือทันทีถ้าขาดทุนหนัก)"
            )

        # ── พิมพ์ผลสรุป ──
        print(f"\n{SEP}")
        print(f"  📌 {ticker}  |  ราคาปัจจุบัน: {current_price:.2f} USD  |  คะแนนรวม: {score_str}")
        print(SEP)

        print(f"\n  ❓ ขึ้นของจริง หรือ แค่เด้งก่อนลงต่อ?")
        print(f"     {verdict}")
        print(f"     {verdict_detail}")

        print(f"\n  💰 ถ้าคุณถืออยู่และ 'กำไร' อยู่:")
        for line in hold_profit.split('\n'):
            print(f"     {line}")

        print(f"\n  📉 ถ้าคุณถืออยู่และ 'ขาดทุน' อยู่:")
        for line in hold_loss.split('\n'):
            print(f"     {line}")

    print(f"\n{SEP}")
    print("\n⚠️  ข้อความข้างต้นเป็นการวิเคราะห์จากข้อมูลตลาดอัตโนมัติ")
    print("    ไม่ใช่คำแนะนำทางการเงิน — การตัดสินใจขึ้นอยู่กับดุลยพินิจของคุณเองเสมอ")


# ── RUN — ต้องรัน Cell 1–4 ก่อน ──
plain_language_summary(df_final, df_pullback, df_momentum, df_rvol)

---
## 🔍 Cell 6 — Next Day Confirmation (ขึ้นจริง หรือแค่ Rebound?)

รันหลังจากวัน Extreme Volume ผ่านไปแล้ว 1 วัน เพื่อตรวจสอบ 3 เงื่อนไข:

| เงื่อนไข | ขึ้นจริง ✅ | Rebound สั้น 🔴 |
|----------|------------|----------------|
| ราคาปิดวัน Extreme Vol | ใกล้ High ของวัน (>60%) | ใกล้ Low ของวัน (<40%) |
| Volume วันถัดไป | ยังสูงกว่าค่าเฉลี่ย (>0.8x) | ร่วงฮวบ (<0.8x avg) |
| ราคาวันถัดไป | ยืนเหนือ Low ของวัน Extreme Vol | หลุดต่ำกว่า Low วัน Extreme Vol |

In [ ]:
def check_next_day_confirmation(tickers, extreme_vol_threshold=2.0):
    """
    ตรวจสอบว่าหุ้นที่มี Extreme Volume วันก่อนหน้า
    วันถัดไปขึ้นจริง หรือแค่ Rebound สั้น

    ตรวจ 3 เงื่อนไข:
    1. Close Position Score: ราคาปิดวัน Extreme Vol อยู่ตรงไหนของ candle
    2. Volume Follow-Through: Volume วันถัดไปยังสูงอยู่ไหม
    3. Structure Hold: ราคาวันถัดไปยืนเหนือ Low วัน Extreme Vol ได้ไหม
    """
    SEP = "─" * 60

    print("\n" + "═" * 60)
    print("  🔍 Next Day Confirmation — ขึ้นจริง หรือแค่ Rebound?")
    print("═" * 60)

    for ticker in tickers:
        try:
            # ดึงข้อมูลรายวัน 30 วันล่าสุด
            df = yf.download(ticker, period='30d', interval='1d', progress=False)
            if df.empty or len(df) < 3:
                print(f"\n⚠️  {ticker}: ข้อมูลไม่พอ")
                continue

            close  = _get_series(df, 'Close',  ticker)
            high   = _get_series(df, 'High',   ticker)
            low    = _get_series(df, 'Low',    ticker)
            volume = _get_series(df, 'Volume', ticker)

            # คำนวณ RVOL รายวัน (เทียบกับ avg 20 วันก่อนหน้า)
            avg_vol_20 = volume.rolling(20).mean().shift(1)  # shift เพื่อไม่ให้ look-ahead
            rvol_daily = volume / avg_vol_20

            # หาวันที่มี Extreme Volume (RVOL >= threshold)
            extreme_days = rvol_daily[rvol_daily >= extreme_vol_threshold].index.tolist()

            if not extreme_days:
                print(f"\n{SEP}")
                print(f"  📌 {ticker}")
                print(f"  ℹ️  ไม่พบวันที่มี Extreme Volume (RVOL >= {extreme_vol_threshold}x) ใน 30 วันล่าสุด")
                continue

            # วิเคราะห์แต่ละวัน Extreme Volume
            print(f"\n{SEP}")
            print(f"  📌 {ticker} — พบ Extreme Volume {len(extreme_days)} วัน")
            print(SEP)

            for ev_date in extreme_days:
                ev_idx = close.index.get_loc(ev_date)

                # ต้องมีวันถัดไปใน data
                has_next_day = ev_idx + 1 < len(close)

                ev_close  = float(close.iloc[ev_idx])
                ev_high   = float(high.iloc[ev_idx])
                ev_low    = float(low.iloc[ev_idx])
                ev_vol    = float(volume.iloc[ev_idx])
                ev_rvol   = float(rvol_daily.iloc[ev_idx])
                ev_avg_vol = float(avg_vol_20.iloc[ev_idx]) if not pd.isna(avg_vol_20.iloc[ev_idx]) else 1

                candle_range = ev_high - ev_low

                # ── เงื่อนไขที่ 1: Close Position Score ──
                # ราคาปิดอยู่ที่ % ไหนของ candle วันนั้น
                if candle_range > 0:
                    close_position = (ev_close - ev_low) / candle_range  # 0=Low, 1=High
                else:
                    close_position = 0.5

                if close_position >= 0.6:
                    cond1_pass = True
                    cond1_label = f"✅ ปิดใกล้ High ({close_position*100:.0f}% ของ candle)"
                    cond1_meaning = "แรงซื้อยังแข็งตอนปิด — คนไม่ขายทิ้ง"
                else:
                    cond1_pass = False
                    cond1_label = f"🔴 ปิดใกล้ Low ({close_position*100:.0f}% ของ candle)"
                    cond1_meaning = "เด้งขึ้นแล้วร่วงกลับตอนปิด — สัญญาณอ่อน"

                # ── เงื่อนไขที่ 2 & 3: ต้องมีวันถัดไป ──
                if has_next_day:
                    nd_close = float(close.iloc[ev_idx + 1])
                    nd_low   = float(low.iloc[ev_idx + 1])
                    nd_vol   = float(volume.iloc[ev_idx + 1])
                    nd_date  = close.index[ev_idx + 1].strftime('%Y-%m-%d')

                    # เงื่อนไขที่ 2: Volume Follow-Through
                    nd_rvol = nd_vol / ev_avg_vol
                    if nd_rvol >= 0.8:
                        cond2_pass = True
                        cond2_label = f"✅ Volume วันถัดไป = {nd_rvol:.2f}x avg (ยังสูงอยู่)"
                        cond2_meaning = "มีแรงซื้อต่อเนื่อง — ไม่ใช่คนแห่เข้าวันเดียวแล้วหนี"
                    else:
                        cond2_pass = False
                        cond2_label = f"🔴 Volume วันถัดไป = {nd_rvol:.2f}x avg (ร่วงฮวบ)"
                        cond2_meaning = "คนแห่เข้าแค่วันเดียวแล้วหายไป — Rebound สั้น"

                    # เงื่อนไขที่ 3: Structure Hold
                    if nd_low >= ev_low:
                        cond3_pass = True
                        cond3_label = f"✅ ราคายืนเหนือ {ev_low:.2f} (Low ของวัน Extreme Vol)"
                        cond3_meaning = "โครงสร้างขาขึ้นยังอยู่ — ฐานแน่น"
                    else:
                        cond3_pass = False
                        cond3_label = f"🔴 ราคาหลุดต่ำกว่า {ev_low:.2f} (Low วัน Extreme Vol)"
                        cond3_meaning = "โครงสร้างพัง — ไม่ใช่การขึ้นจริง"

                    passed = sum([cond1_pass, cond2_pass, cond3_pass])

                    # ── Verdict ──
                    if passed == 3:
                        verdict = "🟢 ขึ้นจริง — ทั้ง 3 เงื่อนไขผ่าน เชื่อถือได้"
                        action  = "→ ถือต่อหรือเพิ่มขนาดได้ ตั้ง Stop ที่ Low วัน Extreme Vol"
                    elif passed == 2:
                        verdict = "🟡 น่าจะขึ้น แต่ยังไม่ชัด — รอยืนยันวันที่ 2"
                        action  = "→ ถือต่อได้ แต่ยังอย่าเพิ่ม position ตั้ง Stop ไว้ก่อน"
                    elif passed == 1:
                        verdict = "🟠 สัญญาณปนกัน — ระวังตัว"
                        action  = "→ ถ้าถืออยู่ให้ตั้ง Stop ใกล้ๆ ถ้ายังไม่เข้าให้รอดูต่อ"
                    else:
                        verdict = "🔴 Rebound สั้น — ทุกเงื่อนไขล้มเหลว"
                        action  = "→ ถ้าถือขาดทุนอยู่ให้ Cut Loss / ถ้ากำไรให้ขายทำกำไร"

                    # ── พิมพ์ผล ──
                    print(f"\n  📅 วัน Extreme Volume: {ev_date.strftime('%Y-%m-%d')}  "
                          f"(RVOL = {ev_rvol:.2f}x)")
                    print(f"     High: {ev_high:.2f}  Low: {ev_low:.2f}  "
                          f"Close: {ev_close:.2f}")
                    print(f"  📅 วันถัดไป: {nd_date}  Close: {nd_close:.2f}")
                    print()
                    print(f"  เงื่อนไข 1 — {cond1_label}")
                    print(f"              ({cond1_meaning})")
                    print(f"  เงื่อนไข 2 — {cond2_label}")
                    print(f"              ({cond2_meaning})")
                    print(f"  เงื่อนไข 3 — {cond3_label}")
                    print(f"              ({cond3_meaning})")
                    print()
                    print(f"  ผ่าน {passed}/3 เงื่อนไข")
                    print(f"  ► {verdict}")
                    print(f"  ► {action}")

                else:
                    # วันล่าสุด — ยังไม่มีวันถัดไป (ตลาดยังไม่ปิด)
                    print(f"\n  📅 วัน Extreme Volume: {ev_date.strftime('%Y-%m-%d')}  "
                          f"(RVOL = {ev_rvol:.2f}x) — นี่คือวันล่าสุด")
                    print(f"     High: {ev_high:.2f}  Low: {ev_low:.2f}  "
                          f"Close: {ev_close:.2f}")
                    print()
                    print(f"  เงื่อนไข 1 — {cond1_label}")
                    print(f"              ({cond1_meaning})")
                    print()
                    print(f"  ⏳ ยังไม่มีข้อมูลวันถัดไป — รอดูเงื่อนไข 2 & 3 วันพรุ่งนี้")
                    if cond1_pass:
                        print(f"  ► เงื่อนไข 1 ผ่านแล้ว — สัญญาณเริ่มต้นดี")
                        print(f"  ► พรุ่งนี้ดูว่า: Volume ยังสูง? และราคาไม่หลุดต่ำกว่า {ev_low:.2f}?")
                    else:
                        print(f"  ► เงื่อนไข 1 ไม่ผ่าน — ระวัง Rebound สั้น")
                        print(f"  ► พรุ่งนี้ดูว่าราคาจะยืนเหนือ {ev_low:.2f} ได้ไหม")

        except Exception as e:
            print(f"\n❌ Error {ticker}: {e}")

    print(f"\n{SEP}")
    print("\n⚠️  ใช้ประกอบการตัดสินใจเท่านั้น ไม่ใช่คำแนะนำทางการเงิน")


# ── CONFIG — ใส่หุ้นที่อยากตรวจสอบ (ไม่จำเป็นต้องเป็นทุกตัวใน STOCK_LIST) ──
# ตั้งค่า extreme_vol_threshold = 2.0 หมายถึง RVOL >= 2x ถือว่า Extreme
CONFIRM_LIST = ['KORU', 'FLKR']   # ← แก้ตรงนี้ได้เลย

check_next_day_confirmation(CONFIRM_LIST, extreme_vol_threshold=2.0)